In [1]:
# combine CMIP6 FWI data from multiple models into a single netCDF file
import xarray as xr
import datetime
import numpy as np
from pathlib import Path
import rioxarray
import cftime
import random
import time

In [2]:
dir = Path("/beegfs/CMIP6/arctic-cmip6/cmip6_fwi/cmip6_fwi/data_release")
# list nc files
files = list(dir.glob("*.nc"))
files.sort()

In [3]:
# if files are era5, split into a different list and drop from the main list
era5_files = [f for f in files if "era5" in f.name]
files = [f for f in files if "era5" not in f.name]

In [ ]:
# open multiple files with xarray
# need to preprocess to include model name and year as coordinates
# example filename structure is cffdrs_<model>_<YYYY>.nc (e.g. cffdrs_CNRM-CM6-1-HR_2007.nc)


def pull_dims_from_source(ds):

    var = list(ds.data_vars)[0]  # just use first var
    src = ds[var].encoding["source"]
    fp_model_id = src.split("/")[-1].split("_")[1]
    # add model to dataset as dimensions using an array with one value
    ds = ds.expand_dims({"model": [fp_model_id]})
    return ds


def preprocess(ds):
    ds = pull_dims_from_source(ds)
    # drop global encoding and attributes that are not needed
    ds.encoding = {}
    ds.attrs = {}
    return ds

In [5]:
cmip6_ds = xr.open_mfdataset(
    files, combine="by_coords", parallel=True, preprocess=preprocess
)
era5_ds = xr.open_mfdataset(
    era5_files, combine="by_coords", parallel=True, preprocess=preprocess
)

In [6]:
# fix the time dimension in era5_ds to be consistent with cmip6_ds
# cmip6_ds time is at noon, era5_ds time is at 0 hour
era5_ds["time"] = [t + datetime.timedelta(hours=12) for t in era5_ds["time"].values]

In [7]:
# Combine cmip6_ds and era5_ds along the "model" dimension, aligning on time
combined_ds = xr.concat([cmip6_ds, era5_ds], dim="model", join="outer")
combined_ds = combined_ds.sortby("time")
combined_ds

/home/jdpaul3/miniconda3/envs/snap-geo/lib/python3.11/site-packages/xarray/core/indexing.py:1624: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/home/jdpaul3/miniconda3/envs/snap-geo/lib/python3.11/site-packages/xarray/core/indexing.py:1624: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.a

<xarray.Dataset> Size: 537GB
Dimensions:  (model: 5, time: 44165, lat: 178, lon: 569)
Coordinates:
  * time     (time) object 353kB 1979-01-01 12:00:00 ... 2099-12-31 12:00:00
  * lat      (lat) float64 1kB 35.0 35.25 35.5 35.75 ... 78.5 78.75 79.0 79.25
  * lon      (lon) float64 5kB -177.0 -176.8 -176.5 ... -35.5 -35.25 -35.0
  * model    (model) object 40B 'CNRM-CM6-1-HR' 'EC-Earth3-Veg' ... 'era5'
Data variables:
    ffmc     (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    dmc      (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    dc       (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    isi      (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    bui      (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    fwi      (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>

In [8]:
# test the dataset; era5 should have NAN for years outside 1979-2023
print(combined_ds["fwi"].sel(model="era5", lat=65, lon=-145, time="1980-01-01").values)
print(combined_ds["fwi"].sel(model="era5", lat=65, lon=-145, time="2080-01-01").values)

[0.9572839]


[nan]


In [9]:
# pick a random file from the era5_files list and the cmip6_files list
# extract the year and model from the filename
# open the file and print original values for lat 65 lon -147 date 01-01
# then compare to the combined dataset values for the same lat lon date and model

test_era5_file = random.choice(era5_files)
test_era5_year = int(test_era5_file.stem.split("_")[2])

test_cmip6_file = random.choice(files)
test_cmip6_year = int(test_cmip6_file.stem.split("_")[2])
test_cmip6_model = test_cmip6_file.stem.split("_")[1]

era5_ds_single = xr.open_dataset(test_era5_file)
cmip6_ds_single = xr.open_dataset(test_cmip6_file)

print(f"Testing ERA5 file: {test_era5_file.name}")
print(
    f"Original ERA5 value: {era5_ds_single['fwi'].sel(lat=65, lon=-147, time=f'{test_era5_year}-01-01').values}"
)
print(
    f"Combined ERA5 value: {combined_ds['fwi'].sel(model='era5', lat=65, lon=-147, time=f'{test_era5_year}-01-01').values}"
)
# wait 3 seconds (print is sometimes jumbled otherwise)

time.sleep(3)

print("\n\n")

print(f"Testing CMIP6 file: {test_cmip6_file.name}")
print(
    f"Original CMIP6 value: {cmip6_ds_single['fwi'].sel(lat=65, lon=-147, time=f'{test_cmip6_year}-01-01').values}"
)
print(
    f"Combined CMIP6 value: {combined_ds['fwi'].sel(model=test_cmip6_model, lat=65, lon=-147, time=f'{test_cmip6_year}-01-01').values}"
)

Testing ERA5 file: cffdrs_era5_2004.nc
Original ERA5 value: [1.9089217]
Combined ERA5 value: [1.9089217]



Testing CMIP6 file: cffdrs_EC-Earth3-Veg_2078.nc
Original CMIP6 value: [0.21503076]
Combined CMIP6 value: [0.21503076]


In [13]:
# drop dim encoding references to source files etc
combined_ds.lon.encoding.pop("source", None)
combined_ds.lon.encoding.pop("original_shape", None)

combined_ds.lat.encoding.pop("source", None)
combined_ds.lat.encoding.pop("original_shape", None)

combined_ds.model.encoding.pop("source", None)
combined_ds.model.encoding.pop("original_shape", None)

combined_ds.time.encoding.pop("source", None)
combined_ds.time.encoding.pop("original_shape", None)
combined_ds.time.encoding.pop("units", None)
combined_ds.time.encoding.pop("calendar", None)

print(combined_ds.lon.encoding)
print(combined_ds.lat.encoding)
print(combined_ds.time.encoding)

{'dtype': dtype('float64'), 'zlib': False, 'szip': False, 'zstd': False, 'bzip2': False, 'blosc': False, 'shuffle': False, 'complevel': 0, 'fletcher32': False, 'contiguous': True, 'chunksizes': None}
{'dtype': dtype('float64'), 'zlib': False, 'szip': False, 'zstd': False, 'bzip2': False, 'blosc': False, 'shuffle': False, 'complevel': 0, 'fletcher32': False, 'contiguous': True, 'chunksizes': None}
{'dtype': dtype('int64'), 'zlib': False, 'szip': False, 'zstd': False, 'bzip2': False, 'blosc': False, 'shuffle': False, 'complevel': 0, 'fletcher32': False, 'contiguous': True, 'chunksizes': None}


In [14]:
# time

# convert time dimension to integer days since 1979-01-01 (using cftime.DatetimeNoLeap for calendar compatibility)
# get start and end dates from combined_ds time coordinate
start_date = combined_ds["time"].values[0]
end_date = combined_ds["time"].values[-1]

days = [(t - start_date).days for t in combined_ds["time"].values]
combined_ds = combined_ds.assign_coords({"time": ("time", days)})

# add metadata to the time dimension
combined_ds["time"].attrs["units"] = "days since 1979-01-01"
combined_ds["time"].attrs["calendar"] = "noleap"
combined_ds["time"].attrs["standard_name"] = "time"
combined_ds["time"].attrs["long_name"] = "time"
combined_ds["time"].attrs["axis"] = "T"
combined_ds["time"].attrs["min_value"] = combined_ds["time"].min().values
combined_ds["time"].attrs["max_value"] = combined_ds["time"].max().values

In [15]:
# variable description dictionary
# from https://cwfis.cfs.nrcan.gc.ca/background/summary/fwi

var_desc = {
    "ffmc": "The Fine Fuel Moisture Code (FFMC) is a numeric rating of the moisture content of litter and other cured fine fuels. This code is an indicator of the relative ease of ignition and the flammability of fine fuel.",
    "dmc": "The Duff Moisture Code (DMC) is a numeric rating of the average moisture content of loosely compacted organic layers of moderate depth. This code gives an indication of fuel consumption in moderate duff layers and medium-size woody material.",
    "dc": "The Drought Code (DC) is a numeric rating of the average moisture content of deep, compact organic layers. This code is a useful indicator of seasonal drought effects on forest fuels and the amount of smoldering in deep duff layers and large logs.",
    "isi": "The Initial Spread Index (ISI) is a numeric rating of the expected rate of fire spread. It is based on wind speed and FFMC. Like the rest of the FWI system components, ISI does not take fuel type into account. Actual spread rates vary between fuel types at the same ISI.",
    "bui": "The Buildup Index (BUI) is a numeric rating of the total amount of fuel available for combustion. It is based on the DMC and the DC. The BUI is generally less than twice the DMC value, and moisture in the DMC layer is expected to help prevent burning in material deeper down in the available fuel.",
    "fwi": "The Fire Weather Index (FWI) is a numeric rating of fire intensity. It is based on the ISI and the BUI, and is used as a general index of fire danger throughout forested areas.",
}

# for each variable, add units attribute (=1 for all variables since they are unitless indices and codes)
# and add _FillValue=np.nan since they are all floats
# and add a description attribute from the var_desc dictionary
for var in combined_ds.data_vars:
    # wipe
    combined_ds[var].attrs = {}
    combined_ds[var].encoding = {}
    # replace
    combined_ds[var].attrs["units"] = "1"
    combined_ds[var].attrs["description"] = var_desc.get(
        var, "No description available."
    )
    combined_ds[var].attrs["_FillValue"] = np.nan

In [16]:
# model encoding dictionary

model_encoding = {
    0: "CNRM-CM6-1-HR",
    1: "EC-Earth3-Veg",
    2: "MPI-ESM1-2-HR",
    3: "MRI-ESM2-0",
    4: "era5",
}

# replace values in the model dimension with integers 0 to n-1 using the model_encoding dictionary
combined_ds = combined_ds.assign_coords(model=[k for k in model_encoding.keys()])

# add units attribute (=1) for the model dimension as well, plus the encodings dictionary
combined_ds["model"].attrs["units"] = "1"
combined_ds["model"].attrs["encoding"] = str(model_encoding)

In [17]:
# global attributes

global_attributes = {
    "attributes": {
        "Conventions": "CF-1.8",  # we aren't 100% CF compliant here, but this is close
        "Title": "CMIP6 CFFDRS Data for the North American Boreal Region",
        "Description": "Canadian Forest Fire Danger Rating System (CFFDRS) calculations of daily estimates for FFMC, DMC, DC, ISI, BUI, and FWI variables over the North American boreal region from 1979-2100. Variables were calculated for historical and ssp585 scenarios of bias-corrected CNRM-CM6-1-HR, EC-Earth3-Veg, MPI-ESM1-2-HR, and MRI-ESM2-0 global climate models, and for baseline ERA5 reanalysis data. The FWI system is a set of numerical ratings and indices based on weather observations that are used worldwide to estimate fire danger in forested areas. The FWI system was developed in Canada and is widely used in North America and other parts of the world. The FWI system is based on daily noon observations of temperature, relative humidity, wind speed, and 24-hour accumulated precipitation. The FWI system consists of six components: the Fine Fuel Moisture Code (FFMC), the Duff Moisture Code (DMC), the Drought Code (DC), the Initial Spread Index (ISI), the Buildup Index (BUI), and the Fire Weather Index (FWI).",
        "Source": "Young, A.M., Littell, J., and Rupp, S., 2025, The influence of fire-fuel feedbacks on boreal forest fire regimes under future climate change: U.S. Geological Survey data release, https://doi.org/10.5066/P1AAMRUF.",
        "Institution": "Scenarios Network for Alaska and Arctic Planning (SNAP), University of Alaska Fairbanks, International Arctic Research Center",
        "URL": "https://www.snap.uaf.edu",
        "Email": "uaf-snap-data-tools@alaska.edu",
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
    },
    "crs": "EPSG:4326",
}

# add global attributes to the dataset
combined_ds.attrs = global_attributes["attributes"]

In [ ]:
# CRS


def add_crs(ds, crs):
    ds = ds.rio.set_spatial_dims("lon", "lat")
    ds = ds.rio.write_crs(crs)  # this creates the "spatial_ref" coordinate
    return ds


combined_ds = add_crs(combined_ds, global_attributes["crs"])

In [19]:
combined_ds

<xarray.Dataset> Size: 537GB
Dimensions:      (model: 5, time: 44165, lat: 178, lon: 569)
Coordinates:
  * lat          (lat) float64 1kB 35.0 35.25 35.5 35.75 ... 78.75 79.0 79.25
  * lon          (lon) float64 5kB -177.0 -176.8 -176.5 ... -35.5 -35.25 -35.0
  * time         (time) int64 353kB 0 1 2 3 4 ... 44160 44161 44162 44163 44164
  * model        (model) int64 40B 0 1 2 3 4
    spatial_ref  int64 8B 0
Data variables:
    ffmc         (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    dmc          (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    dc           (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    isi          (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    bui          (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    fwi          (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.8
    Title:        CMIP6 CFFDRS Data for the North American Boreal Region
    Description:  Canadian Forest Fire Danger Rating System (CFFDRS) calculat...
    Source:       Young, A.M., Littell, J., and Rupp, S., 2025, The influence...
    Institution:  Scenarios Network for Alaska and Arctic Planning (SNAP), Un...
    URL:          https://www.snap.uaf.edu
    Email:        uaf-snap-data-tools@alaska.edu
    start_date:   1979-01-01
    end_date:     2099-12-31

In [20]:
# set up output dir
output_dir = Path(
    "/beegfs/CMIP6/arctic-cmip6/cmip6_fwi/cmip6_fwi/data_release/combined"
)
# if it doesn't exist, make it
output_dir.mkdir(parents=True, exist_ok=True)

In [21]:
# save combined_ds to netcdf, one file for each variable in the dataset
# each file will be ~89GB - an appropriate size for Rasdaman ingestion

for var in combined_ds.data_vars:
    output_file = output_dir / f"cmip6_{var}_combined.nc"
    print(f"Saving {var} to {output_file}")
    combined_ds[[var]].to_netcdf(output_file, mode="w")

Saving ffmc to /beegfs/CMIP6/arctic-cmip6/cmip6_fwi/cmip6_fwi/data_release/combined/cmip6_ffmc_combined.nc


Saving dmc to /beegfs/CMIP6/arctic-cmip6/cmip6_fwi/cmip6_fwi/data_release/combined/cmip6_dmc_combined.nc
Saving dc to /beegfs/CMIP6/arctic-cmip6/cmip6_fwi/cmip6_fwi/data_release/combined/cmip6_dc_combined.nc
Saving isi to /beegfs/CMIP6/arctic-cmip6/cmip6_fwi/cmip6_fwi/data_release/combined/cmip6_isi_combined.nc
Saving bui to /beegfs/CMIP6/arctic-cmip6/cmip6_fwi/cmip6_fwi/data_release/combined/cmip6_bui_combined.nc
Saving fwi to /beegfs/CMIP6/arctic-cmip6/cmip6_fwi/cmip6_fwi/data_release/combined/cmip6_fwi_combined.nc
